# 🕸️ Graph-RAG VLM Pipeline
**Breakdown:**
- **Visual Retriever:** `openai/clip-vit-large-patch14` (CLIP ViT-L/14)
- **Generator (Multimodal LLM):** `Qwen/Qwen2.5-VL-3B-Instruct` (4-bit quantized)
- **Text Embedder:** `sentence-transformers/all-MiniLM-L6-v2`




## 📦 Cell 1 — Install Dependencies

In [ ]:
%pip install -q transformers==4.51.3 accelerate bitsandbytes qwen-vl-utils pdf2image docling pdfplumber networkx sentence-transformers matplotlib fastapi uvicorn pyngrok python-multipart nest_asyncio pillow numpy requests

!apt-get update -qq
!apt-get install -y -qq poppler-utils


## 🧠 Cell 2 — Load All Models

In [ ]:
import torch, gc, os, re, numpy as np, warnings
warnings.filterwarnings('ignore')

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Monkey-patch for colpali if still installed
try:
    from transformers.integrations.peft import _MOE_TARGET_MODULE_MAPPING
    if 'llava' not in _MOE_TARGET_MODULE_MAPPING:
        _MOE_TARGET_MODULE_MAPPING['llava'] = {}
except (ImportError, AttributeError):
    pass

from transformers import (
    CLIPModel, CLIPProcessor,
    Qwen2_5_VLForConditionalGeneration, AutoProcessor,
    BitsAndBytesConfig,
)
from sentence_transformers import SentenceTransformer

# ── 1. CLIP-ViT-Large  (visual retriever, ~430 MB) ──────
print("[1/3] Loading CLIP-ViT-Large (visual retriever)...")
CLIP_MODEL_ID = "openai/clip-vit-large-patch14"
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID,
                     torch_dtype=torch.float16).to("cuda").eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
print(f"  ✅ CLIP ready  "
      f"(VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB)")

# ── 2. Qwen2.5-VL-7B Generator (4-bit) ───────────────────────────────────────
print("[2/3] Loading Qwen2.5-VL-7B-Instruct (4-bit)...")
# Qwen2.5-VL produces GARBAGE (random tokens, "addCriterion", Chinese text, instant EOS)
# when run in float16: its activations overflow fp16 range -> NaN/inf logits.
# Use bfloat16 on Ampere+ GPUs; on older GPUs (T4) fall back to float32 compute.
_major = torch.cuda.get_device_capability()[0]
if _major >= 8 and torch.cuda.is_bf16_supported():
    QWEN_DTYPE = torch.bfloat16
else:
    QWEN_DTYPE = torch.float32          # T4 / V100: never fp16 for Qwen2.5-VL
print(f"  GPU capability {_major}.x -> Qwen dtype = {QWEN_DTYPE}")

try:
    import flash_attn  # noqa: F401
    _attn = "flash_attention_2" if (_major >= 8 and QWEN_DTYPE == torch.bfloat16) else "sdpa"
except ImportError:
    _attn = "sdpa"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=QWEN_DTYPE,
)
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=QWEN_DTYPE,
    device_map="auto",
    attn_implementation=_attn,
).eval()
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print(f"  ✅ Qwen2.5-VL-7B ready  "
      f"(VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB)")



# ── 3. BGE-Base Text Embedder (CPU, ~436 MB) ─────────────────────────────────
print("[3/3] Loading BGE-Base-en-v1.5 (text embedder)...")
text_embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
print("  ✅ BGE-Base ready")

print(f"\n🎉 All models loaded!  Total VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# init steps

## 📄 Cell 3 — PDF Ingestion: Images + Text

In [ ]:
from pdf2image import convert_from_path
from PIL import Image
import pdfplumber


def extract_pages(pdf_path: str, dpi: int = 150):
    """
    Returns:
        images : list[PIL.Image]   – one per page
        texts  : list[str]         – extracted text per page (may be empty)
    """
    print(f"📄 Converting PDF → images  (dpi={dpi}) …")
    images = convert_from_path(pdf_path, dpi=dpi)

    print(f"📝 Extracting text from {len(images)} pages …")
    texts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw = page.extract_text()
            texts.append((raw or "").strip())

    print(f"✅ Extracted {len(images)} pages  "
          f"({sum(1 for t in texts if t)} with text, "
          f"{sum(1 for t in texts if not t)} image-only)")
    return images, texts


print("extract_pages() defined ✅")

## 🔢 Cell 4 — Embedding Functions (Visual + Text)

In [ ]:
from pdf2image import convert_from_path
from PIL import Image
import pdfplumber

def extract_pages(pdf_path: str, dpi: int = 150):
    print(f"📄 Converting PDF → images (dpi={dpi}) …")
    images = convert_from_path(pdf_path, dpi=dpi)
    print(f"📝 Extracting text from {len(images)} pages …")
    texts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            texts.append((page.extract_text() or "").strip())
    print(f"✅ {len(images)} pages  "
          f"({sum(1 for t in texts if t)} with text)")
    return images, texts


def compute_visual_embeddings(images: list, batch_size: int = 8) -> torch.Tensor:
    """
    CLIP image encoder → [n_pages, 768] CLS embeddings.
    Batch_size=8 is fine since CLIP-ViT-Large only uses ~430 MB.
    """
    print("🔢 Computing visual embeddings with CLIP …")
    all_embs = []

    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i : i + batch_size]
            inputs = clip_processor(images=batch, return_tensors="pt",
                                    padding=True).to("cuda")
            # embs = clip_model.get_image_features(**inputs)  # [B, 768]
            outputs = clip_model.get_image_features(**inputs)

            # Handle both cases safely
            if hasattr(outputs, "pooler_output"):
                embs = outputs.pooler_output
            elif isinstance(outputs, tuple):
                embs = outputs[0]
            else:
                embs = outputs


            embs = embs / embs.norm(dim=-1, keepdim=True)   # L2-normalise
            all_embs.append(embs.cpu())
            del inputs, embs
            torch.cuda.empty_cache()
            print(f"  Pages {i+1}–{min(i+batch_size, len(images))} ✓")

    embeddings = torch.cat(all_embs, dim=0)
    print(f"✅ Visual embeddings: {embeddings.shape}")
    return embeddings


def compute_text_embeddings(texts: list) -> torch.Tensor:
    print("📊 Computing text embeddings …")
    processed = [t if t else "[image-only page]" for t in texts]
    embs = text_embedder.encode(processed, convert_to_tensor=True,
                                 show_progress_bar=True, batch_size=32)
    print(f"✅ Text embeddings: {embs.shape}")
    return embs.cpu()

print("Ingestion functions defined ✅")

## cell 9

In [ ]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


def compute_element_embeddings(elements, visual_batch_size=8):
    """
    Generate text embeddings with BGE and visual embeddings with CLIP
    for document elements extracted by Docling.
    """

    # 1. Text embeddings
    print("Embedding individual element text...")

    texts = [
        str(element.get("text", "") or "")
        for element in elements
    ]

    text_vectors = text_embedder.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    for element, vector in zip(elements, text_vectors):
        element["text_embedding"] = vector.detach().cpu().float()

    # 2. Visual embeddings
    print(f"Embedding visual crops: {len(elements)}")

    visual_elements = [
        element
        for element in elements
        if element.get("crop_image") is not None
    ]

    for start in tqdm(
        range(0, len(visual_elements), visual_batch_size),
        desc="CLIP batches",
    ):
        batch_elements = visual_elements[
            start : start + visual_batch_size
        ]

        images = [
            element["crop_image"].convert("RGB")
            for element in batch_elements
        ]

        inputs = clip_processor(
            images=images,
            return_tensors="pt",
            padding=True,
        )

        # Move inputs to the CLIP model's device
        device = next(clip_model.parameters()).device
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            outputs = clip_model.get_image_features(**inputs)

            # Transformers versions may return either a tensor
            # or a BaseModelOutputWithPooling object.
            if isinstance(outputs, torch.Tensor):
                vectors = outputs
            elif hasattr(outputs, "pooler_output"):
                vectors = outputs.pooler_output
            elif hasattr(outputs, "image_embeds"):
                vectors = outputs.image_embeds
            elif hasattr(outputs, "last_hidden_state"):
                vectors = outputs.last_hidden_state[:, 0, :]
            else:
                raise TypeError(
                    "Unexpected CLIP output type: "
                    f"{type(outputs).__name__}"
                )

            vectors = F.normalize(vectors.float(), dim=-1).cpu()

        for element, vector in zip(batch_elements, vectors):
            element["visual_embedding"] = vector

    # Elements without images must remain valid text-only elements.
    for element in elements:
        if element.get("crop_image") is None:
            element["visual_embedding"] = None

    print(
        f"✅ Embeddings complete: {len(elements)} elements, "
        f"{len(visual_elements)} with visual crops."
    )

    return elements


print("Element embedding function defined successfully.")

## 🕸️ Cell 5 — Build Multi-Modal Knowledge Graph

In [ ]:
import networkx as nx
import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity


def build_knowledge_graph(
    elements,
    text_sim_threshold=0.78,
    visual_sim_threshold=0.85,
):
    """Build an element-level graph using text and visual similarity."""

    G = nx.Graph()
    num_elements = len(elements)

    # Add one node per document element
    for idx, element in enumerate(elements):
        node_id = element.get("id", idx)

        node_data = {
            "element_type": element.get("element_type", "unknown"),
            "page_num": element.get("page_num", 1),
            "text": element.get("text", ""),
            "bbox": element.get("bbox"),
            "reading_order": element.get("reading_order", idx),
            "text_emb": element.get(
                "text_embedding", element.get("text_emb")
            ),
            "visual_emb": element.get(
                "visual_embedding", element.get("visual_emb")
            ),
            "crop_image": element.get("crop_image"),
        }

        G.add_node(node_id, **node_data)

    # Connect consecutive elements in reading order
    ordered_elements = sorted(
        enumerate(elements),
        key=lambda pair: (
            pair[1].get("page_num", 1),
            pair[1].get("reading_order", pair[0]),
        ),
    )

    for i in range(len(ordered_elements) - 1):
        _, current = ordered_elements[i]
        _, next_element = ordered_elements[i + 1]

        current_id = current.get("id", elements.index(current))
        next_id = next_element.get("id", elements.index(next_element))

        G.add_edge(
            current_id,
            next_id,
            edge_type="next_element",
            weight=1.0,
        )

    # Build pairwise similarity edges
    for i in range(num_elements):
        element_i = elements[i]
        node_i = element_i.get("id", i)

        for j in range(i + 1, num_elements):
            element_j = elements[j]
            node_j = element_j.get("id", j)

            # Text similarity
            text_i = element_i.get(
                "text_embedding", element_i.get("text_emb")
            )
            text_j = element_j.get(
                "text_embedding", element_j.get("text_emb")
            )

            if text_i is not None and text_j is not None:
                if isinstance(text_i, torch.Tensor):
                    text_i = text_i.detach().cpu().float().numpy()
                if isinstance(text_j, torch.Tensor):
                    text_j = text_j.detach().cpu().float().numpy()

                sim = float(
                    cosine_similarity(
                        text_i.reshape(1, -1),
                        text_j.reshape(1, -1),
                    )[0, 0]
                )

                if sim >= text_sim_threshold:
                    G.add_edge(
                        node_i,
                        node_j,
                        edge_type="text_similar",
                        weight=sim,
                    )

            # Visual similarity
            visual_i = element_i.get(
                "visual_embedding", element_i.get("visual_emb")
            )
            visual_j = element_j.get(
                "visual_embedding", element_j.get("visual_emb")
            )

            if visual_i is not None and visual_j is not None:
                if isinstance(visual_i, torch.Tensor):
                    visual_i = visual_i.detach().cpu().float().numpy()
                if isinstance(visual_j, torch.Tensor):
                    visual_j = visual_j.detach().cpu().float().numpy()

                sim = float(
                    cosine_similarity(
                        visual_i.reshape(1, -1),
                        visual_j.reshape(1, -1),
                    )[0, 0]
                )

                if sim >= visual_sim_threshold:
                    G.add_edge(
                        node_i,
                        node_j,
                        edge_type="visual_similar",
                        weight=sim,
                    )

    print(
        f"✅ Graph built: {G.number_of_nodes()} nodes, "
        f"{G.number_of_edges()} edges"
    )

    return G

## 📊 Cell 6 — Graph Visualisation Helpers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import torch


def visualize_graph(
    G: nx.Graph,
    title: str = "Knowledge Graph",
    highlight_nodes: list = None,
):
    """Visualize the element-level knowledge graph with color-coded edges."""

    if G.number_of_nodes() == 0:
        print("The graph is empty. Nothing to visualize.")
        return

    plt.figure(figsize=(16, 10))

    # Compute node positions
    pos = nx.spring_layout(G, seed=42, k=2.5)

    # Categorize edges according to the graph builder
    seq_e = [
        (u, v)
        for u, v, d in G.edges(data=True)
        if d.get("edge_type") == "next_element"
    ]

    vis_e = [
        (u, v)
        for u, v, d in G.edges(data=True)
        if d.get("edge_type") == "visual_similar"
    ]

    txt_e = [
        (u, v)
        for u, v, d in G.edges(data=True)
        if d.get("edge_type") == "text_similar"
    ]

    # Highlight retrieved elements
    highlight_nodes = set(highlight_nodes or [])

    node_colors = [
        "#F39C12" if n in highlight_nodes else "#2980B9"
        for n in G.nodes()
    ]

    # Draw nodes
    nx.draw_networkx_nodes(
        G,
        pos,
        node_color=node_colors,
        node_size=900,
        alpha=0.95,
    )

    # Label nodes using element type and page number
    nx.draw_networkx_labels(
        G,
        pos,
        labels={
            n: (
                f"{str(G.nodes[n].get('element_type', 'UNK'))[:3]}-"
                f"{G.nodes[n].get('page_num', '?')}"
            )
            for n in G.nodes()
        },
        font_size=9,
        font_color="white",
        font_weight="bold",
    )

    # Draw sequential / reading-order edges
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=seq_e,
        edge_color="#BDC3C7",
        width=1.5,
        alpha=0.6,
        style="dashed",
    )

    # Draw visual similarity edges
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=vis_e,
        edge_color="#E74C3C",
        width=2.5,
        alpha=0.8,
    )

    # Draw text similarity edges
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=txt_e,
        edge_color="#2ECC71",
        width=2.5,
        alpha=0.8,
    )

    # Legend
    legend = [
        mpatches.Patch(
            color="#BDC3C7",
            label="Next Element"
        ),
        mpatches.Patch(
            color="#E74C3C",
            label="Visual Similarity"
        ),
        mpatches.Patch(
            color="#2ECC71",
            label="Text Similarity"
        ),
        mpatches.Patch(
            color="#F39C12",
            label="Retrieved Element"
        ),
        mpatches.Patch(
            color="#2980B9",
            label="Regular Element"
        ),
    ]

    plt.legend(
        handles=legend,
        loc="upper left",
        fontsize=9,
        framealpha=0.9,
    )

    plt.title(title, fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def plot_similarity_heatmap(
    sim_matrix: torch.Tensor,
    title: str = "Element Similarity Heatmap",
):
    """Display a heatmap of pairwise element similarities."""

    # Convert tensor to a NumPy array safely
    data = sim_matrix.detach().cpu().float().numpy()

    if data.size == 0:
        print("The similarity matrix is empty. Nothing to visualize.")
        return

    num_elements = len(data)

    fig, ax = plt.subplots(
        figsize=(
            max(6, num_elements // 2),
            max(5, num_elements // 2),
        )
    )

    im = ax.imshow(
        data,
        cmap="Blues",
        vmin=0,
        vmax=1,
        aspect="auto",
    )

    ax.set_xticks(range(num_elements))
    ax.set_yticks(range(num_elements))

    ax.set_xticklabels(
        [f"E{i + 1}" for i in range(num_elements)],
        fontsize=8,
    )

    ax.set_yticklabels(
        [f"E{i + 1}" for i in range(num_elements)],
        fontsize=8,
    )

    plt.colorbar(im, ax=ax, label="Similarity")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Element")
    ax.set_ylabel("Element")

    plt.tight_layout()
    plt.show()


print("Visualization helpers defined successfully.")

## 🔍 Cell 7 — Graph-Guided RAG (Retrieval + Generation)

In [ ]:
import torch.nn.functional as F
from qwen_vl_utils import process_vision_info
from IPython.display import display, Markdown, HTML
import re
import numpy as np
from PIL import Image
import torch
import gc

def clean_latex(text: str) -> str:
    text = re.sub(r'\\\[\s*(.*?)\s*\\\]', r'$$\1$$', text, flags=re.DOTALL)
    text = re.sub(r'\\\(\s*(.*?)\s*\\\)', r'$\1$',   text, flags=re.DOTALL)
    text = re.sub(r'(?<!\\)\[\s*(.*?=.*?)\s*\]', r'$$\1$$', text)
    return text

def _inline_math(text: str) -> str:
    """Bold + inline $math$ → MathJax \\(math\\)."""
    text = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', text)
    text = re.sub(
        r'(?<!\$)\$(?!\$)(.*?)(?<!\$)\$(?!\$)',
        lambda m: f'\\({m.group(1)}\\)',
        text
    )
    return text

# ── Markdown table → HTML ─────────────────────────────────────────────────────

def _is_separator_row(row: str) -> bool:
    """Detect |---|---| alignment rows."""
    return bool(re.match(r'^[\|\s\-:]+$', row))

def parse_markdown_table(block: str) -> str:
    """Convert a markdown table string into a styled HTML table."""
    lines = [l.strip() for l in block.strip().split('\n') if l.strip()]
    if len(lines) < 2:
        return f'<pre>{block}</pre>'

    html_rows = []
    header_done = False

    for line in lines:
        if not line.startswith('|'):
            continue
        if _is_separator_row(line):
            header_done = True
            continue

        cells = [c.strip() for c in line.strip('|').split('|')]
        tag = 'td'
        style = 'padding:8px 14px; border:1px solid #ccd; font-size:14px;'

        if not header_done:
            tag = 'th'
            style += 'background:#dce8f8; font-weight:600; text-align:center;'

        cell_html = ''.join(
            f'<{tag} style="{style}">{_inline_math(c)}</{tag}>'
            for c in cells
        )
        html_rows.append(f'<tr>{cell_html}</tr>')

    if not html_rows:
        return f'<pre>{block}</pre>'

    return f"""
    <div style="overflow-x:auto; margin:14px 0;">
      <table style="border-collapse:collapse; font-family:Arial,sans-serif;
                    width:100%; font-size:14px;">
        {''.join(html_rows)}
      </table>
    </div>"""

# ── LaTeX table → plain HTML fallback ────────────────────────────────────────

def parse_latex_table_fallback(latex: str) -> str:
    """
    Best-effort LaTeX table → HTML.
    Strips all LaTeX commands and renders raw cell text.
    Used only when Qwen ignores the markdown instruction.
    """
    caption_match = re.search(r'\\caption\{(.*?)\}', latex, re.DOTALL)
    caption = caption_match.group(1).strip() if caption_match else ""

    tabular_match = re.search(
        r'\\begin\{tabular\}.*?\}(.*?)\\end\{tabular\}',
        latex, re.DOTALL
    )
    if not tabular_match:
        return f'<pre style="white-space:pre-wrap">{latex}</pre>'

    body = tabular_match.group(1)
    body = re.sub(r'\\hline', '', body)

    body = re.sub(r'\\multicolumn\{\d+\}\{[^}]*\}\{(.*?)\}', r'\1', body,
                  flags=re.DOTALL)
    body = re.sub(r'\\multirow\{[^}]+\}\{[^}]*\}\{(.*?)\}', r'\1', body,
                  flags=re.DOTALL)
    body = re.sub(r'\\begin\{tabular\}.*?\}(.*?)\\end\{tabular\}', r'\1',
                  body, flags=re.DOTALL)
    body = re.sub(r'\\[a-zA-Z]+\*?(\{[^}]*\})*', ' ', body)

    rows_raw = re.split(r'\\\\', body)
    rows = [r.strip() for r in rows_raw if r.strip()]

    html_rows = []
    for i, row in enumerate(rows):
        cols = [c.strip() for c in row.split('&') if c.strip()]
        if not cols:
            continue
        tag = 'th' if i == 0 else 'td'
        base = 'padding:8px 14px; border:1px solid #ccd; font-size:13px;'
        extra = 'background:#dce8f8; font-weight:600;' if i == 0 else ''
        cell_html = ''.join(
            f'<{tag} style="{base}{extra}">{_inline_math(c)}</{tag}>'
            for c in cols
        )
        html_rows.append(f'<tr>{cell_html}</tr>')

    caption_html = (
        f'<caption style="caption-side:bottom;padding:8px;'
        f'font-style:italic;color:#555;">{caption}</caption>'
        if caption else ''
    )

    return f"""
    <div style="overflow-x:auto; margin:14px 0;">
      <table style="border-collapse:collapse; font-family:Arial,sans-serif; width:100%;">
        {caption_html}{''.join(html_rows)}
      </table>
    </div>"""

# ── Segment splitter ──────────────────────────────────────────────────────────

def _split_segments(text: str) -> list:
    """
    Returns list of (kind, content):
      'latex_table'    → \\begin{table}...\\end{table}
      'md_table'       → consecutive lines starting with |
      'text'           → everything else
    """
    segments = []

    latex_pat = re.compile(r'(\\begin\{table\}.*?\\end\{table\})', re.DOTALL)
    parts = latex_pat.split(text)

    for part in parts:
        if part.startswith(r'\begin{table}'):
            segments.append(('latex_table', part))
            continue

        lines = part.split('\n')
        buffer      = []
        table_buf   = []
        in_md_table = False

        def flush_text():
            if buffer:
                segments.append(('text', '\n'.join(buffer)))
                buffer.clear()

        def flush_md():
            if table_buf:
                segments.append(('md_table', '\n'.join(table_buf)))
                table_buf.clear()

        for line in lines:
            if line.strip().startswith('|'):
                if not in_md_table:
                    flush_text()
                    in_md_table = True
                table_buf.append(line)
            else:
                if in_md_table:
                    flush_md()
                    in_md_table = False
                buffer.append(line)

        flush_md()
        flush_text()

    return segments

# ── Master renderer ───────────────────────────────────────────────────────────

def render_answer(answer: str):
    answer = clean_latex(answer)
    segments = _split_segments(answer)
    html_parts = []

    for kind, content in segments:

        if kind == 'latex_table':
            html_parts.append(parse_latex_table_fallback(content))

        elif kind == 'md_table':
            html_parts.append(parse_markdown_table(content))

        else:  # plain text
            for line in content.split('\n'):
                s = line.strip()
                if s.startswith('$$') and s.endswith('$$') and len(s) > 4:
                    html_parts.append(
                        f'<div class="math-block">\\[{s[2:-2].strip()}\\]</div>'
                    )
                elif s.startswith('### '):
                    html_parts.append(f'<h4>{_inline_math(s[4:])}</h4>')
                elif s.startswith('## '):
                    html_parts.append(f'<h3>{_inline_math(s[3:])}</h3>')
                elif s.startswith('# '):
                    html_parts.append(f'<h2>{_inline_math(s[2:])}</h2>')
                elif s.startswith('**') and s.endswith('**'):
                    html_parts.append(
                        f'<p><strong>{_inline_math(s[2:-2])}</strong></p>'
                    )
                elif s.startswith(('- ', '* ')):
                    html_parts.append(f'<li>{_inline_math(s[2:])}</li>')
                elif s == '':
                    html_parts.append('<br>')
                else:
                    html_parts.append(f'<p>{_inline_math(line)}</p>')

    body = '\n'.join(html_parts)

    display(HTML(f"""
    <script>
      if (typeof MathJax === 'undefined') {{
        var s = document.createElement('script');
        s.src = 'https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js';
        s.async = true; document.head.appendChild(s);
      }} else {{ MathJax.typesetPromise && MathJax.typesetPromise(); }}
    </script>
    <div style="font-family:'Segoe UI',Arial,sans-serif; font-size:15px;
                line-height:1.8; padding:16px 20px; background:#f8f9fa;
                border-left:4px solid #4A90D9; border-radius:6px;
                margin:10px 0; color:#1a1a2e;">
      <div style="font-weight:700; color:#4A90D9; margin-bottom:10px;">🤖 Answer</div>
      {body}
    </div>
    <style>
      .math-block {{text-align:center;margin:12px 0;padding:8px;
                   background:#eef2ff;border-radius:4px;font-size:16px;}}
      li {{margin:4px 0;}} p {{margin:6px 0;}}
    </style>"""))



import torch
import torch.nn.functional as F
from qwen_vl_utils import process_vision_info
from PIL import Image
import gc


def graph_guided_rag(
    query,
    G,
    visual_embeddings=None,
    top_k=4,
    hops=1,
    n_seeds=3,
    show_graph=True,
):
    """Retrieve relevant document elements and generate an evidence-grounded answer."""

    if G is None:
        raise ValueError(
            "Knowledge graph is None. Index a PDF before querying."
        )

    if G.number_of_nodes() == 0:
        raise ValueError("The knowledge graph contains no elements.")

    print(f"\n{'=' * 62}")
    print(f"QUERY: {query}")
    print("=" * 62)
    print("PIPELINE_STATUS:understanding")

    node_ids = list(G.nodes())

    # Drop low-information fragments ("Reported by:", "Digitally signed: ...")
    # so they never crowd out real content.
    def _is_noise(nid):
        d = G.nodes[nid]
        t = str(d.get("text") or "").strip()
        if d.get("crop_image") is not None:
            return False
        if d.get("element_type") in ("heading", "title", "table", "caption", "formula", "code"):
            return len(t) < 3
        low = t.lower()
        return (
            len(t) < 25
            or low.startswith("digitally signed")
            or low.startswith("reported by")
        )

    noise_set = {n for n in node_ids if _is_noise(n)}
    node_ids = [n for n in node_ids if n not in noise_set] or list(G.nodes())
    if len(node_ids) == len(G.nodes()):
        noise_set = set()

    # ---------------------------------------------------------
    # 1. Text retrieval using BGE
    # ---------------------------------------------------------
    print("PIPELINE_STATUS:text_search")
    query_text_emb = text_embedder.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True,
    ).detach().cpu().float().reshape(1, -1)

    text_scores = {}

    for node_id in node_ids:
        emb = G.nodes[node_id].get("text_emb")

        if emb is None:
            text_scores[node_id] = -1.0
            continue

        if not isinstance(emb, torch.Tensor):
            emb = torch.as_tensor(emb)

        emb = emb.detach().cpu().float().reshape(1, -1)

        score = F.cosine_similarity(
            query_text_emb,
            emb,
            dim=-1,
        ).item()

        # Give validated corrections a relevance boost when the query
        # overlaps with their question/correction text.
        if G.nodes[node_id].get("element_type") == "correction":
            query_terms = set(re.findall(r"\w+", query.lower()))
            node_text = str(G.nodes[node_id].get("text", "")).lower()
            node_terms = set(re.findall(r"\w+", node_text))

            overlap = len(query_terms & node_terms) / max(len(query_terms), 1)
            score += 0.35 * overlap

        text_scores[node_id] = score

    text_ranked = sorted(
        node_ids,
        key=lambda n: text_scores[n],
        reverse=True,
    )

    # ---------------------------------------------------------
    # 2. Visual retrieval using CLIP
    # ---------------------------------------------------------
    print("PIPELINE_STATUS:visual_search")
    visual_ids = [
        node_id
        for node_id in node_ids
        if G.nodes[node_id].get("visual_emb") is not None
    ]

    visual_scores = {}
    visual_ranked = []

    if visual_ids:
        clip_device = next(clip_model.parameters()).device

        with torch.inference_mode():
            # Always define query_inputs in this function.
            query_inputs = clip_processor(
                text=[query],
                return_tensors="pt",
                padding=True,
                truncation=True,
            )

            query_inputs = {
                key: value.to(clip_device)
                for key, value in query_inputs.items()
            }

            query_outputs = clip_model.get_text_features(
                **query_inputs
            )

            # Support Tensor and Transformers model-output formats.
            if isinstance(query_outputs, torch.Tensor):
                query_visual_emb = query_outputs

            elif (
                getattr(query_outputs, "text_embeds", None)
                is not None
            ):
                query_visual_emb = query_outputs.text_embeds

            elif (
                getattr(query_outputs, "pooler_output", None)
                is not None
            ):
                query_visual_emb = query_outputs.pooler_output

            elif (
                getattr(query_outputs, "last_hidden_state", None)
                is not None
            ):
                query_visual_emb = (
                    query_outputs.last_hidden_state[:, 0, :]
                )

            else:
                raise TypeError(
                    "Cannot extract CLIP text features from "
                    f"{type(query_outputs).__name__}. "
                    f"Available attributes: "
                    f"{[a for a in dir(query_outputs) if not a.startswith('_')]}"
                )

            query_visual_emb = F.normalize(
                query_visual_emb.float(),
                dim=-1,
            ).detach().cpu()

        for node_id in visual_ids:
            emb = G.nodes[node_id]["visual_emb"]

            if not isinstance(emb, torch.Tensor):
                emb = torch.as_tensor(emb)

            emb = emb.detach().cpu().float().reshape(1, -1)

            if emb.shape[-1] != query_visual_emb.shape[-1]:
                raise ValueError(
                    f"CLIP embedding dimension mismatch for node "
                    f"{node_id}: query={query_visual_emb.shape[-1]}, "
                    f"element={emb.shape[-1]}. Use the same CLIP "
                    "model for query text and image embeddings."
                )

            visual_scores[node_id] = F.cosine_similarity(
                query_visual_emb,
                emb,
                dim=-1,
            ).item()

        visual_ranked = sorted(
            visual_ids,
            key=lambda n: visual_scores[n],
            reverse=True,
        )

        del query_inputs, query_outputs, query_visual_emb

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ---------------------------------------------------------
    # 3. Reciprocal Rank Fusion
    # ---------------------------------------------------------
    k_rrf = 60
    scores = {}

    for rank, node_id in enumerate(text_ranked, start=1):
        scores[node_id] = scores.get(node_id, 0.0) + (
            1.0 / (k_rrf + rank)
        )

    for rank, node_id in enumerate(visual_ranked, start=1):
        scores[node_id] = scores.get(node_id, 0.0) + (
            1.0 / (k_rrf + rank)
        )

    seeds = sorted(
        scores,
        key=lambda n: scores[n],
        reverse=True,
    )[:max(1, n_seeds)]

    print("Seed elements:", seeds)

    # ---------------------------------------------------------
    # 4. Expand through graph relationships
    # ---------------------------------------------------------
    print("PIPELINE_STATUS:graph_context")
    candidate_scores = dict(scores)
    frontier = set(seeds)
    visited = set(seeds)

    for _ in range(max(0, hops)):
        next_frontier = set()

        for node_id in frontier:
            for neighbor in G.neighbors(node_id):
                if neighbor in visited or neighbor in noise_set:
                    continue

                edge = G.edges[node_id, neighbor]
                edge_type = edge.get("edge_type", "unknown")

                if edge_type not in {
                    "next_element",
                    "text_similar",
                    "visual_similar",
                    "user_correction",
                }:
                    continue

                candidate_scores[neighbor] = max(
                    candidate_scores.get(neighbor, 0.0),
                    scores.get(node_id, 0.0) * 0.85,
                )

                next_frontier.add(neighbor)

        visited.update(next_frontier)
        frontier = next_frontier

    selected = sorted(
        candidate_scores,
        key=lambda n: candidate_scores[n],
        reverse=True,
    )[:max(1, top_k)]

    def _order_key(n):
        d = G.nodes[n]
        return (d.get("page_num", 1), d.get("reading_order", 0))

    # Broad questions ("tell me about this pdf", "summarize") need the document
    # skeleton (titles / headings / tables / key paragraphs), not 8 random hits.
    OVERVIEW_RE = re.compile(
        r"\b(summar\w*|overview|gist|tl;?dr)\b"
        r"|\b(about|describe|explain|contents?|what\s+is)\b.*\b(pdf|document|report|file|paper)\b",
        re.I,
    )
    if OVERVIEW_RE.search(query):
        ordered = sorted([n for n in G.nodes() if n not in noise_set], key=_order_key)
        skeleton = [
            n for n in ordered
            if G.nodes[n].get("element_type") in ("title", "heading", "caption")
        ]
        body = sorted(
            [n for n in ordered if n not in skeleton],
            key=lambda n: len(str(G.nodes[n].get("text") or "")),
            reverse=True,
        )[:8]
        selected = list(dict.fromkeys(skeleton[:16] + body + selected))[:max(top_k, 24)]
        print("Overview query -> using document skeleton + key content")

    # Present evidence in reading order so the model sees a coherent document.
    selected = sorted(selected, key=_order_key)

    print(f"Selected {len(selected)} elements:")

    for node_id in selected:
        data = G.nodes[node_id]
        print(
            f"  {node_id} | "
            f"{data.get('element_type', 'unknown')} | "
            f"page {data.get('page_num', '?')}"
        )

    # ---------------------------------------------------------
    # 5. Visualize retrieval
    # ---------------------------------------------------------
    if show_graph:
        visualize_graph(
            G,
            title=f"Element retrieval: {query[:50]}",
            highlight_nodes=selected,
        )

    # ---------------------------------------------------------
    # 6. Prepare retrieved text and images
    # ---------------------------------------------------------
    print("PIPELINE_STATUS:prepare_context")
    evidence_parts = []
    retrieved_images = []

    for node_id in selected:
        data = G.nodes[node_id]

        evidence_parts.append(
            f"[Element ID: {node_id}]\n"
            f"[Type: {data.get('element_type', 'unknown')}]\n"
            f"[Page: {data.get('page_num', '?')}]\n"
            f"Content:\n{str(data.get('text', ''))[:1500]}"
        )

        crop = data.get("crop_image")

        if crop is not None:
            crop = crop.convert("RGB")
            width, height = crop.size
            scale = min(1.0, 1024 / max(width, height))

            if scale < 1.0:
                crop = crop.resize(
                    (
                        max(1, int(width * scale)),
                        max(1, int(height * scale)),
                    ),
                    Image.LANCZOS,
                )

            retrieved_images.append(crop)

    evidence_text = "\n\n---\n\n".join(evidence_parts)



    # Collect validated corrections relevant to this question.
    corrections = []

    for node_id in selected:
        node = G.nodes[node_id]

        if node.get("element_type") == "correction":
            correction_text = str(node.get("text", "")).strip()

            if correction_text:
                corrections.append({
                    "node_id": str(node_id),
                    "text": correction_text,
                })

    # Also inspect all correction nodes, since a correction can be
    # relevant even when ordinary retrieval doesn't select it.
    query_terms = set(re.findall(r"\w+", query.lower()))

    for node_id, node in G.nodes(data=True):
        if node.get("element_type") != "correction":
            continue

        correction_text = str(node.get("text", "")).strip()
        correction_terms = set(re.findall(r"\w+", correction_text.lower()))

        if query_terms and query_terms.issubset(correction_terms):
            if not any(c["node_id"] == str(node_id) for c in corrections):
                corrections.append({
                    "node_id": str(node_id),
                    "text": correction_text,
                })

    correction_context = ""

    if corrections:
        correction_context = (
            "VALIDATED USER CORRECTIONS — PRIORITIZE THESE:\n"
            "These corrections were explicitly submitted by the user. "
            "For a question addressed by a correction, use the corrected "
            "information instead of conflicting original document claims. "
            "Do not repeat a superseded answer.\n\n"
            + "\n\n".join(
                f"[Correction node: {c['node_id']}]\n{c['text']}"
                for c in corrections
            )
            + "\n\n"
        )

    print("Corrections included in prompt:", len(corrections))
    print(correction_context)


    content = [
        {
            "type": "text",
            "text": (
                 "Answer in English using the supplied evidence.\n"
                  "Never invent missing information.\n\n"
                  + correction_context
                  + "Retrieved document evidence:\n"
                  + evidence_text
                  + "\n\nQuestion: "
                  + query
            ),
        }
    ]

    # content.extend(
    #     {"type": "image", "image": img}
    #     for img in retrieved_images
    # )


    SYSTEM_PROMPT = """
    You are MIRAS, a precise document question-answering assistant.

    Answer questions using only the supplied document evidence and images.

    Rules:
    1. For questions like "What is this PDF about?", explain the document's
      purpose, major topics, and important findings using the available evidence.
    2. For questions about patients or people, extract only details explicitly
      present in the evidence. Use a numbered list or Markdown table.
    3. Write complete, grammatical sentences. Never return a partial sentence.
    4. Never output internal tool calls, tool_call tags, or internal reasoning.
    5. Never invent missing information. Clearly state when evidence is insufficient.
    6. Cite page numbers when they are available in the evidence.
    7. Return only the final answer in clean Markdown.
    8. Always answer in English unless the user writes in another language.
    9. Ignore any disclaimers about the document being test data; still answer
       from its content, mentioning the disclaimer at most once.
    """

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": content,
        },
    ]


    # ---------------------------------------------------------
    # 7. Generate answer with Qwen
    # ---------------------------------------------------------
    print("PIPELINE_STATUS:generating")

    qwen_device = next(qwen_model.parameters()).device

    text_in = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    img_in, vid_in = process_vision_info(messages)

    print("\n========== MULTIMODAL INPUT DEBUG ==========")
    print("Images extracted:", 0 if img_in is None else len(img_in))
    print("Videos extracted:", 0 if vid_in is None else len(vid_in))
    print("Prompt tail:")
    print(text_in[-2500:])

    inputs = qwen_processor(
        text=[text_in],
        images=img_in,
        videos=vid_in,
        padding=True,
        return_tensors="pt",
    ).to(qwen_device)



    print("Input token count:", inputs.input_ids.shape[1])
    print("Input IDs tail:", inputs.input_ids[0, -40:].tolist())
    print(
        "Decoded input tail:",
        repr(qwen_processor.tokenizer.decode(
            inputs.input_ids[0, -150:],
            skip_special_tokens=False
        ))
    )

    MAX_NEW_TOKENS = 1024
    tokenizer = qwen_processor.tokenizer

    # Cast any floating inputs (pixel_values) to the model dtype.
    for k, v in list(inputs.items()):
        if torch.is_tensor(v) and v.is_floating_point():
            inputs[k] = v.to(qwen_model.dtype)

    def _generate():
        with torch.inference_mode():
            return qwen_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                repetition_penalty=1.0,
                eos_token_id=[tokenizer.convert_tokens_to_ids("<|im_end|>"),
                              tokenizer.eos_token_id],
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )

    def _garbage(txt):
        t = txt.strip()
        if len(t) < 15:
            return True
        if "addCriterion" in t or "<|im_start|>" in t:
            return True
        letters = [c for c in t if c.isalpha()]
        if letters:
            non_latin = sum(1 for c in letters if ord(c) > 0x024F)
            if non_latin / len(letters) > 0.3:
                return True
        return False

    gen_ids = _generate()
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    raw_answer = tokenizer.decode(trimmed[0], skip_special_tokens=False)
    answer = tokenizer.decode(trimmed[0], skip_special_tokens=True).strip()

    token_count = len(trimmed[0])
    print("\n========== GENERATION DIAGNOSTICS ==========")
    print("Model dtype:", qwen_model.dtype)
    print("Generated tokens:", token_count)
    print("Reached output limit:", token_count >= MAX_NEW_TOKENS)
    print("Raw output:", repr(raw_answer[:500]))
    print("============================================")

    print("PIPELINE_STATUS:checking")
    if _garbage(answer):
        print("WARNING: model output looks invalid -> using evidence-based fallback.")
        lines = []
        for node_id in selected:
            d = G.nodes[node_id]
            t = " ".join(str(d.get("text") or "").split())
            if t:
                lines.append(f"- **{d.get('element_type','element')} (page {d.get('page_num','?')})**: {t[:400]}")
        answer = (
            "The language model returned an unusable answer, so here is the "
            "retrieved evidence from the document:\n\n" + "\n".join(lines)
        )

    # ---------------------------------------------------------
    # 8. Collect source pages
    # ---------------------------------------------------------
    source_pages = sorted({
        int(G.nodes[node_id]["page_num"]) - 1
        for node_id in selected
        if G.nodes[node_id].get("page_num") is not None
    })

    # ---------------------------------------------------------
    # 9. Cleanup and return
    # ---------------------------------------------------------
    del inputs, gen_ids

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    return clean_latex(answer), retrieved_images, source_pages, selected


print("Element-level Graph RAG ready.")


## ⚙️ Cell 8 — Master Pipeline Runner

In [ ]:
pdf_images = pdf_texts = visual_embeddings = None
text_embeddings = knowledge_graph = vis_sim_matrix = txt_sim_matrix = None

pdf_images = None
document_elements = None
docling_document = None

visual_embeddings = None
knowledge_graph = None


def run_full_pipeline(pdf_path, dpi=150, show_heatmaps=False):
    global pdf_images, document_elements, docling_document
    global visual_embeddings, knowledge_graph

    print("\n" + "=" * 62)
    print(f"Indexing: {os.path.basename(pdf_path)}")
    print("=" * 62)

    # 1. Parse document into elements.
    pdf_images, document_elements, docling_document = (
        extract_document_elements(pdf_path, dpi=dpi)
    )

    # 2. Generate embeddings per element.
    visual_embeddings = compute_element_embeddings(
        document_elements
    )

    # 3. Build graph with element nodes.
    knowledge_graph = build_knowledge_graph(
        document_elements,
        text_sim_threshold=0.78,
        visual_sim_threshold=0.80,
    )

    # 4. Visualize the element graph.
    visualize_graph(
        knowledge_graph,
        title=f"Element Graph — {os.path.basename(pdf_path)}",
    )

    print("\nPipeline completed.")
    print(f"Document pages: {len(pdf_images)}")
    print(f"Element nodes: {knowledge_graph.number_of_nodes()}")
    print(f"Graph edges: {knowledge_graph.number_of_edges()}")

    return knowledge_graph


print("Element-level ingestion pipeline ready.")


print("run_full_pipeline() defined ✅")

# =============================   Phase 4 Entity graph =============================

import os
import re
import json
import torch
import networkx as nx
import numpy as np

from PIL import Image
from pdf2image import convert_from_path
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_options = PdfPipelineOptions(
    generate_page_images=True,
    generate_picture_images=True,
    do_table_structure=True,
)

docling_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

CROP_DIR = "docling_element_crops"
os.makedirs(CROP_DIR, exist_ok=True)


def _label_value(item):
    label = getattr(item, "label", "unknown")
    return str(getattr(label, "value", label)).lower()


def _get_page_number(item):
    provenance = getattr(item, "prov", None) or []

    if provenance:
        return int(provenance[0].page_no)

    return 1


def _get_bbox(item):
    provenance = getattr(item, "prov", None) or []

    if not provenance:
        return None

    bbox = getattr(provenance[0], "bbox", None)

    if bbox is None:
        return None

    return {
        "left": float(bbox.l),
        "top": float(bbox.t),
        "right": float(bbox.r),
        "bottom": float(bbox.b),
        "coordinate_origin": str(
            getattr(bbox, "coord_origin", "unknown")
        ),
    }


def _get_element_text(item, doc, element_type):
    # Tables should retain their cell structure.
    if element_type == "table":
        try:
            return item.export_to_markdown(doc=doc)
        except Exception:
            pass

    text = getattr(item, "text", None)

    if text:
        return str(text).strip()

    # Preserve a searchable fallback for pictures without descriptions.
    if element_type in ("picture", "figure", "chart"):
        return f"[Visual element: {element_type}]"

    return ""


def _get_element_image(item, doc):
    """
    Extract an element's own image when Docling exposes one.
    Returns None when an element has no usable image.
    """
    try:
        image = item.get_image(doc)

        if image is None:
            return None

        if isinstance(image, Image.Image):
            return image.convert("RGB")

        # Handle image-reference wrappers when available.
        pil_image = getattr(image, "pil_image", None)

        if isinstance(pil_image, Image.Image):
            return pil_image.convert("RGB")

    except Exception:
        pass

    return None


def extract_document_elements(pdf_path, dpi=150):
    """
    Returns:
        page_images: full-page PIL images
        elements: one record per detected document element
        doc: original DoclingDocument
    """
    print("Converting PDF with Docling...")

    result = docling_converter.convert(pdf_path)
    doc = result.document

    print("Rendering original pages...")
    page_images = convert_from_path(pdf_path, dpi=dpi)

    elements = []
    reading_order = 0

    for item, level in doc.iterate_items():
        label = _label_value(item)

        # Exclude repetitive page furniture from retrieval.
        if label in {"page_header", "page_footer"}:
            continue

        page_num = _get_page_number(item)

        # Normalize Docling's labels to our graph node types.
        if "table" in label:
            element_type = "table"
        elif "picture" in label or "figure" in label:
            element_type = "picture"
        elif "formula" in label:
            element_type = "formula"
        elif "title" in label:
            element_type = "title"
        elif "section_header" in label:
            element_type = "heading"
        elif "list_item" in label:
            element_type = "list_item"
        elif "caption" in label:
            element_type = "caption"
        elif "code" in label:
            element_type = "code"
        else:
            element_type = "paragraph"

        text = _get_element_text(item, doc, element_type)

        # Skip empty non-visual elements.
        crop = _get_element_image(item, doc)

        if not text and crop is None:
            continue

        element_id = f"element_{reading_order:06d}"
        crop_path = None

        if crop is not None:
            crop_path = os.path.join(
                CROP_DIR, f"{element_id}.png"
            )
            crop.save(crop_path)

        elements.append({
            "id": element_id,
            "element_type": element_type,
            "source_label": label,
            "text": text,
            "document_id": os.path.basename(pdf_path),
            "page_num": page_num,
            "bbox": _get_bbox(item),
            "reading_order": reading_order,
            "crop_path": crop_path,
            "crop_image": crop,
            "docling_level": int(level),
        })

        reading_order += 1

    print(f"Pages: {len(page_images)}")
    print(f"Detected elements: {len(elements)}")

    from collections import Counter
    print("Element types:", Counter(
        e["element_type"] for e in elements
    ))

    return page_images, elements, doc


print("Docling extraction functions ready.")

## 📥 Cell 9A — Option A : Download Sample PDF (Attention Is All You Need)
_Run **either** Cell 9A **or** Cell 9B — not both._

In [ ]:
import os
import requests
import subprocess

pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_filename = "attention_paper.pdf"

# Verify Poppler is available before starting the pipeline
if not os.path.exists("/usr/bin/pdfinfo"):
    raise RuntimeError(
        "Poppler is missing. Run the installation cell first:\n"
        "!apt-get install -y poppler-utils"
    )

# Download the PDF if it does not already exist
if not os.path.exists(pdf_filename):
    print("⬇️ Downloading sample PDF...")

    response = requests.get(
        pdf_url,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    # Ensure the downloaded response is actually a PDF
    if not response.content.startswith(b"%PDF"):
        raise ValueError(
            "The downloaded file is not a valid PDF. "
            "Check the URL or the server response."
        )

    with open(pdf_filename, "wb") as f:
        f.write(response.content)

    print(f"✅ Saved as {pdf_filename}")

else:
    print(f"✅ {pdf_filename} already exists — skipping download.")

# Verify the PDF can be read by Poppler
subprocess.run(
    ["pdfinfo", pdf_filename],
    check=True,
    capture_output=True,
    text=True,
)

print("\n" + "=" * 60)
print(f"Indexing: {pdf_filename}")
print("=" * 60)

run_full_pipeline(pdf_filename)

In [ ]:
# 🔥 Step 2: Interactive loop
print("\n" + "="*70)
print("💬 Graph-RAG Interactive Session Started")
print("Type 'quit' to exit")
print("="*70)

while True:
    query = input("\n🧠 Your Question → ").strip()

    if query.lower() in ["quit", "exit", "q"]:
        print("👋 Exiting session...")
        break

    if not query:
        continue

    answer, source_images, source_pages, _selected = graph_guided_rag(
        query             = query,
        G                 = knowledge_graph,
        visual_embeddings = visual_embeddings,
        top_k             = 2,
        hops              = 1,
        n_seeds           = 2,
        show_graph        = False
    )

    # print("\n" + "─"*60)
    # print("🤖 ANSWER")
    # print("─"*60)
    render_answer(answer)

    print(f"\n📄 Source Pages → {[p+1 for p in source_pages]}")

## 📤 Cell 9B — Option B : Upload Your Own PDF
_Run **either** Cell 9A **or** Cell 9B — not both._

In [ ]:
# 📤 Cell 9B — Upload Your Own PDF
# Run either Cell 9A or Cell 9B, not both.

from google.colab import files

print("📂 Select a PDF to upload:")
uploaded = files.upload()

if uploaded:
    pdf_filename = next(iter(uploaded))

    if not pdf_filename.lower().endswith(".pdf"):
        raise ValueError("Please upload a PDF file.")

    print(f"\n✅ Uploaded: {pdf_filename}")

    # Run the complete indexing pipeline.
    pipeline_result = run_full_pipeline(pdf_filename)

    # If the pipeline returns a result dictionary, use it.
    # Otherwise, use the global variables assigned by the pipeline.
    if isinstance(pipeline_result, dict):
        knowledge_graph = pipeline_result.get(
            "knowledge_graph", knowledge_graph
        )
        document_elements = pipeline_result.get(
            "document_elements", document_elements
        )
        visual_embeddings = pipeline_result.get(
            "visual_embeddings", visual_embeddings
        )

    # Validate the graph before allowing queries.
    if knowledge_graph is None:
        raise RuntimeError(
            "Indexing finished, but knowledge_graph is None. "
            "Update run_full_pipeline() to assign the graph and "
            "return it, as shown below."
        )

    if knowledge_graph.number_of_nodes() == 0:
        raise RuntimeError(
            "The knowledge graph contains no nodes."
        )

    print("\n✅ PDF indexed successfully.")
    print(f"   Elements: {len(document_elements)}")
    print(f"   Graph nodes: {knowledge_graph.number_of_nodes()}")
    print(f"   Graph edges: {knowledge_graph.number_of_edges()}")

else:
    print("❌ No file uploaded.")

## 💬 Cell 11 — Interactive Query Loop
Type questions one at a time. Enter `quit` to stop.

In [ ]:
# 💬 Cell 11 — Interactive Query Loop

if knowledge_graph is None:
    raise RuntimeError(
        "No knowledge graph found. Run Cell 9A or Cell 9B first."
    )

if knowledge_graph.number_of_nodes() == 0:
    raise RuntimeError(
        "The knowledge graph is empty. Re-index your PDF."
    )

print("=" * 65)
print("💬 Interactive Graph-RAG Session Started")
print("Type 'quit', 'exit', or 'q' to stop.")
print("=" * 65)

while True:
    query = input("\n🧠 Your question ► ").strip()

    if query.lower() in ("quit", "exit", "q", "stop"):
        print("👋 Session ended.")
        break

    if not query:
        continue

    try:
        answer, source_images, source_pages, _selected = graph_guided_rag(
            query=query,
            G=knowledge_graph,
            visual_embeddings=visual_embeddings,
            top_k=4,
            hops=1,
            n_seeds=3,
            show_graph=False,
        )

        print("\n" + "-" * 65)
        render_answer(answer)

        if source_pages:
            print(
                f"\n📄 Source pages: "
                f"{[page + 1 for page in source_pages]}"
            )
        else:
            print("\n📄 No source pages returned.")

        print("-" * 65)

    except Exception as exc:
        print(f"\n❌ Query failed: {type(exc).__name__}: {exc}")
        print("You can try another query or type 'quit' to exit.")

# FastAPI Backend setup

### Persistent knowledge-graph store (ChromaDB) — run before the FastAPI cell

In [ ]:
# ============================================================
# PERSISTENT KNOWLEDGE-GRAPH STORE (ChromaDB)
# Stores ONLY the knowledge graph (nodes, embeddings, edges, crops).
# Chat history is never stored.
# ============================================================
%pip install -q chromadb

import os, io, json, time, shutil
import numpy as np
import torch
import networkx as nx
import chromadb
from PIL import Image

# Colab's /content is wiped when the runtime is recycled.
# To survive that too, mount Drive and use e.g. "/content/drive/MyDrive/miras_store".
PERSIST_DIR = os.environ.get("MIRAS_STORE", "/content/miras_store")
CROP_STORE = os.path.join(PERSIST_DIR, "crops")
os.makedirs(CROP_STORE, exist_ok=True)

PDF_STORE = os.path.join(PERSIST_DIR, "pdfs")  # original PDF, so the viewer works after refresh
os.makedirs(PDF_STORE, exist_ok=True)


def pdf_path(doc_id):
    return os.path.join(PDF_STORE, f"{doc_id}.pdf")


def save_pdf(doc_id, src_path):
    if not os.path.exists(pdf_path(doc_id)):
        shutil.copyfile(src_path, pdf_path(doc_id))

_chroma = chromadb.PersistentClient(path=os.path.join(PERSIST_DIR, "chroma"))

def _collection(name, space):
    return _chroma.get_or_create_collection(name=name, metadata={"hnsw:space": space})

col_text     = _collection("kg_text_nodes",   "cosine")  # BGE text embedding per node
col_visual   = _collection("kg_visual_nodes", "cosine")  # CLIP crop embedding per node
col_manifest = _collection("kg_manifest",     "l2")      # per-doc edges + metadata, active pointer
col_feedback = _collection("kg_user_feedback", "l2")


_BATCH = 500
ACTIVE_ID = "__active__"


def _to_list(vec):
    if isinstance(vec, torch.Tensor):
        vec = vec.detach().cpu().float().numpy()
    return np.asarray(vec, dtype=np.float32).reshape(-1).tolist()


def _get_all(col, where, include):
    """Page through every record that matches `where`."""
    out = {k: [] for k in ["ids"] + include}
    offset = 0
    while True:
        res = col.get(where=where, include=include, limit=_BATCH, offset=offset)
        ids = res["ids"]
        if len(ids) == 0:
            break
        out["ids"].extend(ids)
        for k in include:
            v = res[k]
            out[k].extend(list(v) if v is not None else [None] * len(ids))
        offset += len(ids)
        if len(ids) < _BATCH:
            break
    return out


def doc_exists(doc_id):
    return len(col_manifest.get(ids=[doc_id])["ids"]) > 0


def set_active_doc(doc_id):
    col_manifest.upsert(
        ids=[ACTIVE_ID], embeddings=[[0.0]],
        metadatas=[{"kind": "active", "doc_id": doc_id}],
        documents=[doc_id],
    )


def get_active_doc_id():
    res = col_manifest.get(ids=[ACTIVE_ID], include=["metadatas"])
    if res["ids"]:
        did = res["metadatas"][0].get("doc_id")
        if did and doc_exists(did):
            return did
    return None


def list_documents():
    res = col_manifest.get(where={"kind": "doc"}, include=["metadatas"])
    docs = [dict(m) for m in res["metadatas"]]
    docs.sort(key=lambda m: m.get("created_at", 0), reverse=True)
    return docs


def delete_document(doc_id):
    col_text.delete(where={"doc_id": doc_id})
    col_visual.delete(where={"doc_id": doc_id})
    col_manifest.delete(ids=[doc_id])
    shutil.rmtree(os.path.join(CROP_STORE, doc_id), ignore_errors=True)
    try:
        os.remove(pdf_path(doc_id))
    except OSError:
        pass
    if get_active_doc_id() is None:
        try:
            col_manifest.delete(ids=[ACTIVE_ID])
        except Exception:
            pass


def persist_graph(doc_id, filename, G, page_sizes, pages, n_elements):
    """Save the knowledge graph. Re-saving the same doc_id overwrites it."""

    # Do not delete the document or its PDF when updating the graph.
    # Existing node IDs are stable, so upsert their records instead.

    crop_dir = os.path.join(CROP_STORE, doc_id)
    os.makedirs(crop_dir, exist_ok=True)

    t_ids, t_embs, t_meta, t_docs = [], [], [], []
    v_ids, v_embs, v_meta = [], [], []
    dim = None

    for nid, d in G.nodes(data=True):
        te = d.get("text_emb")
        if te is not None and dim is None:
            dim = len(_to_list(te))

    for nid, d in G.nodes(data=True):
        nid = str(nid)
        te = d.get("text_emb")
        t_vec = _to_list(te) if te is not None else [0.0] * (dim or 768)

        has_crop = d.get("crop_image") is not None
        if has_crop:
            d["crop_image"].convert("RGB").save(os.path.join(crop_dir, f"{nid}.png"))

        meta = {
            "doc_id": doc_id,
            "element_type": str(d.get("element_type", "unknown")),
            "page_num": int(d.get("page_num") or 1),
            "reading_order": int(d.get("reading_order") or 0),
            "bbox_json": json.dumps(d.get("bbox")),
            "has_crop": bool(has_crop),
            "has_text_emb": te is not None,
        }
        t_ids.append(f"{doc_id}::{nid}")
        t_embs.append(t_vec)
        t_meta.append(meta)
        t_docs.append(str(d.get("text") or ""))

        ve = d.get("visual_emb")
        if ve is not None:
            v_ids.append(f"{doc_id}::{nid}")
            v_embs.append(_to_list(ve))
            v_meta.append({"doc_id": doc_id, "node_id": nid})

    for i in range(0, len(t_ids), _BATCH):
        s = slice(i, i + _BATCH)
        col_text.upsert(
            ids=t_ids[s],
            embeddings=t_embs[s],
            metadatas=t_meta[s],
            documents=t_docs[s],
        )
    for i in range(0, len(v_ids), _BATCH):
        s = slice(i, i + _BATCH)
        col_visual.upsert(
            ids=v_ids[s],
            embeddings=v_embs[s],
            metadatas=v_meta[s],
        )

    edges = [
        [str(u), str(v), str(e.get("edge_type", "unknown")), float(e.get("weight", 1.0))]
        for u, v, e in G.edges(data=True)
    ]
    payload = {
        "edges": edges,
        "page_sizes": {str(k): [float(v[0]), float(v[1])] for k, v in (page_sizes or {}).items()},
    }
    col_manifest.upsert(
        ids=[doc_id], embeddings=[[0.0]],
        documents=[json.dumps(payload)],
        metadatas=[{
            "kind": "doc", "doc_id": doc_id, "filename": str(filename),
            "pages": int(pages), "elements": int(n_elements),
            "nodes": int(G.number_of_nodes()), "edges": len(edges),
            "created_at": float(time.time()),
        }],
    )
    set_active_doc(doc_id)
    print(f"💾 Saved knowledge graph '{filename}' ({doc_id}): "
          f"{G.number_of_nodes()} nodes, {len(edges)} edges")


def load_graph(doc_id):
    """Rebuild the exact networkx graph used by graph_guided_rag. None if not stored."""
    man = col_manifest.get(ids=[doc_id], include=["metadatas", "documents"])
    if not man["ids"]:
        return None
    info = man["metadatas"][0]
    payload = json.loads(man["documents"][0])

    texts = _get_all(col_text, {"doc_id": doc_id}, ["embeddings", "metadatas", "documents"])
    vis = _get_all(col_visual, {"doc_id": doc_id}, ["embeddings", "metadatas"])
    vis_by_node = {
        m["node_id"]: torch.tensor(np.asarray(e, dtype=np.float32))
        for m, e in zip(vis["metadatas"], vis["embeddings"])
    }

    G = nx.Graph()
    crop_dir = os.path.join(CROP_STORE, doc_id)

    for rid, emb, meta, doc in zip(
        texts["ids"], texts["embeddings"], texts["metadatas"], texts["documents"]
    ):
        nid = rid.split("::", 1)[1]
        crop = None
        if meta.get("has_crop"):
            p = os.path.join(crop_dir, f"{nid}.png")
            if os.path.exists(p):
                with Image.open(p) as im:
                    crop = im.convert("RGB")
        G.add_node(
            nid,
            element_type=meta["element_type"],
            page_num=int(meta["page_num"]),
            text=doc or "",
            bbox=json.loads(meta.get("bbox_json") or "null"),
            reading_order=int(meta["reading_order"]),
            text_emb=(torch.tensor(np.asarray(emb, dtype=np.float32))
                      if meta.get("has_text_emb") else None),
            visual_emb=vis_by_node.get(nid),
            crop_image=crop,
        )

    for u, v, etype, w in payload["edges"]:
        if u in G and v in G:
            G.add_edge(u, v, edge_type=etype, weight=float(w))

    page_sizes = {int(k): (v[0], v[1]) for k, v in payload["page_sizes"].items()}

    print(f"📂 Loaded knowledge graph '{info.get('filename')}' ({doc_id}): "
          f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return {
        "graph": G,
        "visual_embeddings": None,   # not used by graph_guided_rag
        "page_sizes": page_sizes,
        "pages": int(info.get("pages", 0)),
        "elements": int(info.get("elements", G.number_of_nodes())),
        "filename": info.get("filename"),
    }

def reset_all():
    """Delete every saved graph, crop and PDF."""
    for d in list_documents():
        delete_document(d["doc_id"])
    try:
        col_manifest.delete(ids=[ACTIVE_ID])
    except Exception:
        pass
    shutil.rmtree(CROP_STORE, ignore_errors=True)
    os.makedirs(CROP_STORE, exist_ok=True)
    for f in os.listdir(PDF_STORE):
        try:
            os.remove(os.path.join(PDF_STORE, f))
        except OSError:
            pass

print(f"Persistent KG store ready at {PERSIST_DIR} "
      f"({len(list_documents())} saved document(s)) ✅")


## run the FastAPI server

In [ ]:
# FastAPI Backend setup

!apt-get update
!pip install -q fastapi uvicorn pyngrok python-multipart nest_asyncio
!apt-get install -y poppler-utils



import os
import json
import asyncio
import queue
import threading
import tempfile
import hashlib
import contextlib
import sys
import nest_asyncio
import uvicorn
import uuid

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, FileResponse
from pydantic import BaseModel
from pyngrok import ngrok


# ============================================================
# CONFIG
# ============================================================

# Which coordinate space are your stored node bboxes in?
#   "image" -> bboxes are in the pixel space of page_images (default)
#   "pdf"   -> bboxes are in PDF points (Docling default); page sizes are
#              then read from the Docling document instead of the images
BBOX_SPACE = "image"


# ============================================================
# APP SETUP
# ============================================================

app = FastAPI(title="Graph-Guided Multimodal RAG API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Restrict to your frontend URL in production
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

state = {
    "graph": None,
    "vis_embs": None,
    "page_sizes": {},
    "ready": False,
    "graph_data_json": None,
    "filename": None,
    "pages": 0,
    "elements": 0,
    "doc_id": None,
}

state_lock = threading.RLock()
stdout_lock = threading.Lock()

STOP = object()


class QueryRequest(BaseModel):
    query: str


from pydantic import BaseModel, Field
from typing import Literal

class FeedbackRequest(BaseModel):
    doc_id: str
    question: str
    answer: str
    correction: str = Field(min_length=5, max_length=5000)
    source_ids: list[str] = Field(default_factory=list, max_length=50)
    rating: Literal["helpful", "correction"] = "correction"


# ============================================================
# JSON SERIALIZATION
# ============================================================

def make_json_safe(value):
    """Convert common Python/NumPy/Pydantic values to JSON-safe values."""

    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if hasattr(value, "model_dump"):
        return make_json_safe(value.model_dump())

    if hasattr(value, "tolist"):
        return make_json_safe(value.tolist())

    if isinstance(value, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [make_json_safe(v) for v in value]

    return str(value)


def node_payload(G, node_id, page_sizes, text_len=500):
    """Single payload shape used for BOTH graph nodes and chat citations."""
    d = G.nodes[node_id]
    page = d.get("page_num")  # 1-based in the graph
    w, h = (page_sizes or {}).get(page, (None, None))

    return {
        "id": str(node_id),
        "name": str(node_id),
        "element_type": d.get("element_type", "unknown"),
        "type": d.get("element_type", "unknown"),
        "page": make_json_safe(page),
        "page_num": make_json_safe(page),
        "text": str(d.get("text") or "")[:text_len],
        "bbox": make_json_safe(d.get("bbox")),
        "bbox_norm": normalize_bbox(d.get("bbox"), (w, h)),
        "page_width": w,
        "page_height": h,
        "reading_order": make_json_safe(d.get("reading_order")),
    }


def serialize_graph(G, page_sizes=None):
    nodes = [node_payload(G, n, page_sizes) for n in G.nodes()]

    links = [
        {
            "source": str(u),
            "target": str(v),
            # Supports the current graph and older edge attributes.
            "type": d.get("type", d.get("edge_type", "unknown")),
            "weight": make_json_safe(d.get("weight", 1.0)),
        }
        for u, v, d in G.edges(data=True)
    ]

    return {"nodes": nodes, "links": links}


def ndjson(data):
    """Encode one newline-delimited JSON event."""
    return json.dumps(
        data,
        ensure_ascii=False,
        default=str
    ) + "\n"


def normalize_bbox(bbox, page_size):
    """Docling bbox dict -> [l, t, r, b] as 0-1 fractions, top-left origin."""
    if not bbox or not page_size or not page_size[0] or not page_size[1]:
        return None
    try:
        W, H = float(page_size[0]), float(page_size[1])
        l, r = float(bbox["left"]), float(bbox["right"])
        t, b = float(bbox["top"]), float(bbox["bottom"])
        origin = str(bbox.get("coordinate_origin", "")).upper()
        if "BOTTOMLEFT" in origin:          # flip y to a top-left origin
            t, b = H - t, H - b
        clamp = lambda v: min(1.0, max(0.0, v))
        return [
            clamp(min(l, r) / W), clamp(min(t, b) / H),
            clamp(max(l, r) / W), clamp(max(t, b) / H),
        ]
    except (KeyError, TypeError, ValueError):
        return None

def compute_page_sizes(page_images, doc, dpi=150):
    """{page_no (1-based): (width, height)} in PDF points (same units as Docling bboxes)."""
    sizes = {}

    # Fallback: convert rendered image pixels back to points
    for i, img in enumerate(page_images):
        sizes[i + 1] = (img.size[0] * 72.0 / dpi, img.size[1] * 72.0 / dpi)

    # Preferred: exact page size from the Docling document
    try:
        for page_no, page in doc.pages.items():
            sizes[int(page_no)] = (float(page.size.width), float(page.size.height))
    except Exception as e:
        print(f"Could not read page sizes from Docling doc, using image-derived sizes: {e}")

    return sizes


# ============================================================
# HEALTH / STATUS
# ============================================================

@app.get("/")
async def root():
    with state_lock:
        return {
            "status": "online",
            "ready": state["ready"],
            "filename": state["filename"],
            "pages": state["pages"],
            "elements": state["elements"],
        }


@app.get("/status")
async def get_status():
    with state_lock:
        return {
            "ready": state["ready"],
            "filename": state["filename"],
            "pages": state["pages"],
            "elements": state["elements"],
            "doc_id": state["doc_id"],
            "nodes": (
                state["graph"].number_of_nodes()
                if state["graph"] is not None
                else 0
            ),
            "edges": (
                state["graph"].number_of_edges()
                if state["graph"] is not None
                else 0
            ),
        }


# ============================================================
# PDF UPLOAD AND INDEXING
# ============================================================

def index_pdf(path):
    """
    Uses the current notebook functions:
      extract_document_elements(path)
      compute_element_embeddings(elements)
      build_knowledge_graph(elements)
    """

    page_images, elements, doc = extract_document_elements(path)

    if not elements:
        raise ValueError("No document elements were extracted from the PDF.")

    visual_embs = compute_element_embeddings(elements)
    G = build_knowledge_graph(elements)

    if G is None or G.number_of_nodes() == 0:
        raise ValueError("The knowledge graph contains no nodes.")

    page_sizes = compute_page_sizes(page_images, doc)
    graph_data = serialize_graph(G, page_sizes)

    return {
        "graph": G,
        "visual_embeddings": visual_embs,
        "graph_data": graph_data,
        "page_sizes": page_sizes,
        "pages": len(page_images),
        "elements": len(elements),
    }


def apply_state(result, filename, doc_id):
    with state_lock:
        state["graph"] = result["graph"]
        state["vis_embs"] = result.get("visual_embeddings")
        state["page_sizes"] = result["page_sizes"]
        state["graph_data_json"] = result["graph_data"]
        state["ready"] = True
        state["filename"] = filename
        state["pages"] = result["pages"]
        state["elements"] = result["elements"]
        state["doc_id"] = doc_id


def restore_doc(doc_id):
    """Load a saved graph from ChromaDB and build the frontend payload. None if missing."""
    result = load_graph(doc_id)
    if result is None:
        return None
    result["graph_data"] = serialize_graph(result["graph"], result["page_sizes"])
    return result


@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...)):
    if file.filename and not file.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Please upload a PDF file."
        )

    path = None
    hasher = hashlib.sha256()

    try:
        # Unique temporary file prevents filename collisions.
        with tempfile.NamedTemporaryFile(
            suffix=".pdf",
            prefix="rag_upload_",
            delete=False,
        ) as temp_file:
            path = temp_file.name

            while chunk := await file.read(1024 * 1024):
                hasher.update(chunk)
                temp_file.write(chunk)

        if os.path.getsize(path) == 0:
            raise HTTPException(
                status_code=400,
                detail="The uploaded file is empty."
            )

        doc_id = hasher.hexdigest()[:16]          # same PDF bytes -> same id
        display_name = os.path.basename(file.filename or "document.pdf")

        # Already indexed before? Load the saved graph instead of re-running Docling/CLIP.
        cached = False
        result = None
        try:
            result = await asyncio.to_thread(restore_doc, doc_id)
        except Exception as e:
            print(f"Could not load saved graph, re-indexing: {e}")
        if result is not None:
            cached = True
            await asyncio.to_thread(set_active_doc, doc_id)
        else:
            # Docling and embedding operations are blocking, so run off-loop.
            result = await asyncio.to_thread(index_pdf, path)
            try:
                await asyncio.to_thread(
                    persist_graph, doc_id, display_name, result["graph"],
                    result["page_sizes"], result["pages"], result["elements"],
                )
            except Exception as e:
                # Never fail the upload just because saving failed.
                print(f"WARNING: could not save knowledge graph: {e}")

        try:
            await asyncio.to_thread(save_pdf, doc_id, path)
        except Exception as e:
            print(f"WARNING: could not store PDF copy: {e}")

        apply_state(result, display_name, doc_id)

        return {
            "status": "success",
            "cached": cached,
            "message": (
                ("Loaded saved knowledge graph: " if cached else "Indexed ")
                + f"{result['elements']} elements from {result['pages']} pages."
            ),
            "doc_id": doc_id,
            "filename": display_name,
            "pages": result["pages"],
            "elements": result["elements"],
            "graphData": result["graph_data"],
        }

    except HTTPException:
        raise

    except Exception as e:
        # Keep the previously indexed document if a new upload fails.
        raise HTTPException(
            status_code=500,
            detail=f"PDF indexing failed: {str(e)}"
        )

    finally:
        await file.close()

        if path and os.path.exists(path):
            try:
                os.remove(path)
            except OSError:
                pass


# ============================================================
# SAVED KNOWLEDGE GRAPHS (restore after frontend refresh)
# ============================================================

@app.on_event("startup")
async def restore_last_session():
    """On server start, reload the last active knowledge graph, if any."""
    try:
        doc_id = await asyncio.to_thread(get_active_doc_id)
        if doc_id:
            result = await asyncio.to_thread(restore_doc, doc_id)
            if result:
                apply_state(result, result["filename"], doc_id)
    except Exception as e:
        print(f"Startup restore skipped: {e}")


@app.get("/graph")
async def get_graph():
    """Call this when the frontend loads/refreshes. Returns the saved graph + node metadata."""
    with state_lock:
        if not state["ready"] or state["graph"] is None:
            return {"ready": False, "graphData": None}
        return {
            "ready": True,
            "doc_id": state["doc_id"],
            "filename": state["filename"],
            "pages": state["pages"],
            "elements": state["elements"],
            "graphData": state["graph_data_json"],
        }


@app.get("/documents")
async def documents():
    with state_lock:
        active = state["doc_id"]
    docs = await asyncio.to_thread(list_documents)
    return {"active": active, "documents": docs}


@app.post("/documents/{doc_id}/load")
async def load_document(doc_id: str):
    result = await asyncio.to_thread(restore_doc, doc_id)
    if result is None:
        raise HTTPException(status_code=404, detail="Saved document not found.")
    await asyncio.to_thread(set_active_doc, doc_id)
    apply_state(result, result["filename"], doc_id)
    return {
        "ready": True, "doc_id": doc_id, "filename": result["filename"],
        "pages": result["pages"], "elements": result["elements"],
        "graphData": result["graph_data"],
    }


@app.delete("/documents/{doc_id}")
async def remove_document(doc_id: str):
    await asyncio.to_thread(delete_document, doc_id)
    with state_lock:
        if state["doc_id"] == doc_id:
            state.update(graph=None, vis_embs=None, page_sizes={}, graph_data_json=None,
                         ready=False, filename=None, pages=0, elements=0, doc_id=None)
    return {"deleted": doc_id}

@app.get("/pdf/{doc_id}")
async def get_pdf(doc_id: str):
    """Serves the stored PDF so the viewer works after a refresh."""
    if not doc_id.isalnum():
        raise HTTPException(status_code=400, detail="Bad document id.")
    p = pdf_path(doc_id)
    if not os.path.exists(p):
        raise HTTPException(status_code=404, detail="PDF not stored for this document.")
    return FileResponse(p, media_type="application/pdf")


@app.post("/reset")
async def reset_everything():
    """Delete all saved graphs/PDFs and clear the active session."""
    await asyncio.to_thread(reset_all)
    with state_lock:
        state.update(graph=None, vis_embs=None, page_sizes={}, graph_data_json=None,
                     ready=False, filename=None, pages=0, elements=0, doc_id=None)
    return {"status": "reset"}

# ============================================================
# QUERY STREAMING
# ============================================================

class QueueOut:
    """Capture printed pipeline progress and forward it to the stream."""

    def __init__(self, original_stdout, output_queue):
        self.original_stdout = original_stdout
        self.output_queue = output_queue
        self.buffer = ""

    def write(self, text):
        self.original_stdout.write(text)
        self.original_stdout.flush()

        self.buffer += text

        while "\n" in self.buffer:
            line, self.buffer = self.buffer.split("\n", 1)
            self._emit(line)

        return len(text)


    def _emit(self, line):
        message = line.strip()

        if not message.startswith("PIPELINE_STATUS:"):
            return

        step = message.split(":", 1)[1].strip()

        status_messages = {
            "understanding": (
                "understanding",
                "Understanding your question...",
            ),
            "text_search": (
                "text_search",
                "Searching relevant document content...",
            ),
            "visual_search": (
                "visual_search",
                "Searching visual document content...",
            ),
            "graph_context": (
                "graph_context",
                "Connecting related information...",
            ),
            "prepare_context": (
                "prepare_context",
                "Preparing relevant document context...",
            ),
            "generating": (
                "generating",
                "Generating your answer...",
            ),
            "checking": (
                "checking",
                "Checking the response...",
            ),
        }

        if step not in status_messages:
            return

        step_name, user_message = status_messages[step]

        self.output_queue.put({
            "type": "progress",
            "step": step_name,
            "status": user_message,
        })


    def flush(self):
        self.original_stdout.flush()

        if self.buffer.strip():
            self._emit(self.buffer)
            self.buffer = ""

    def isatty(self):
        return False


def run_rag_worker(query, G, visual_embeddings, page_sizes, output_queue):
    """Run graph_guided_rag in a worker thread and queue the result."""

    try:
        # stdout is process-global; serialize captures between concurrent queries.
        with stdout_lock:
            with contextlib.redirect_stdout(
                QueueOut(sys.stdout, output_queue)
            ):
                answer, _retrieved_images, _source_pages, selected_ids = graph_guided_rag(
                    query=query,
                    G=G,
                    visual_embeddings=visual_embeddings,
                    top_k=12,
                    hops=1,
                    n_seeds=8,
                    show_graph=False,
                )

        output_queue.put({
            "type": "progress",
            "step": "finalizing",
            "status": "Preparing your final answer..."
        })

        output_queue.put({
            "type": "answer",
            "answer": str(answer),
            # One object per retrieved element: id, type, page, text, bbox, page size
            "sources": [
                node_payload(G, n, page_sizes, text_len=1000)
                for n in selected_ids
            ],
        })

    except Exception as e:
        output_queue.put({
            "error": str(e),
            "status": f"Pipeline Error: {str(e)}",
        })

    finally:
        output_queue.put(STOP)


@app.post("/query")
async def query_pdf(req: QueryRequest):
    query = req.query.strip()

    if not query:
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    with state_lock:
        if not state["ready"] or state["graph"] is None:
            raise HTTPException(
                status_code=400,
                detail="No PDF has been indexed yet. Upload a PDF first."
            )

        # Snapshot the current document state for this request.
        G = state["graph"]
        visual_embeddings = state["vis_embs"]
        page_sizes = state["page_sizes"]

    output_queue = queue.Queue()

    async def event_generator():
        loop = asyncio.get_running_loop()

        worker = loop.run_in_executor(
            None,
            run_rag_worker,
            query,
            G,
            visual_embeddings,
            page_sizes,
            output_queue,
        )

        try:
            while True:
                try:
                    event = output_queue.get_nowait()
                except queue.Empty:
                    if worker.done():
                        break

                    await asyncio.sleep(0.1)
                    continue

                if event is STOP:
                    break

                yield ndjson(event)

            # Propagate unexpected executor errors, if any.
            await worker

            # Drain events queued immediately before worker completion.
            while not output_queue.empty():
                event = output_queue.get_nowait()

                if event is not STOP:
                    yield ndjson(event)

        except asyncio.CancelledError:
            # Client disconnected. The worker thread may finish in the background.
            raise

        except Exception as e:
            yield ndjson({
                "error": str(e),
                "status": f"Streaming Error: {str(e)}",
            })

    return StreamingResponse(
        event_generator(),
        media_type="application/x-ndjson",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )



@app.post("/feedback")
async def submit_feedback(req: FeedbackRequest):
    correction = req.correction.strip()

    # Validate the active document and resolve frontend source IDs
    # to the actual NetworkX node IDs.
    with state_lock:
        if not state["ready"] or state["graph"] is None:
            raise HTTPException(
                status_code=400,
                detail="No indexed document is available.",
            )

        if req.doc_id != state["doc_id"]:
            raise HTTPException(
                status_code=409,
                detail="The document has changed. Please refresh.",
            )

        G = state["graph"]
        page_sizes = dict(state["page_sizes"])
        filename = state["filename"]
        pages = state["pages"]
        elements = state["elements"]

        node_lookup = {str(n): n for n in G.nodes()}

        source_nodes = list(dict.fromkeys(
            node_lookup[sid]
            for sid in req.source_ids
            if sid in node_lookup
        ))

        if not source_nodes:
            raise HTTPException(
                status_code=400,
                detail="No valid source elements were supplied.",
            )

        feedback_id = str(uuid.uuid4())
        correction_node_id = f"correction_{feedback_id}"

        source_pages = [
            G.nodes[n].get("page_num")
            for n in source_nodes
            if G.nodes[n].get("page_num") is not None
        ]
        correction_page = int(source_pages[0]) if source_pages else 1

        # Include the original question in the node text so that
        # future queries can retrieve the correction by meaning.
        correction_node_text = (
            f"Question: {req.question.strip()}\n"
            f"Validated correction: {correction}"
        )

        # Use the same text embedding model as the original graph.
        correction_embedding = text_embedder.encode(
            correction_node_text,
            convert_to_tensor=True,
            normalize_embeddings=True,
        ).detach().cpu().float().reshape(-1)

        # 1. Physically add a correction node to NetworkX.
        G.add_node(
            correction_node_id,
            element_type="correction",
            page_num=correction_page,
            text=correction_node_text,
            bbox=None,
            reading_order=0,
            text_emb=correction_embedding,
            visual_emb=None,
            crop_image=None,
            is_user_correction=True,
            original_question=req.question,
            original_answer=req.answer,
            source_ids=[str(n) for n in source_nodes],
        )

        # 2. Physically add edges between the correction and its
        # cited source nodes. The graph is undirected.
        for source_node in source_nodes:
            G.add_edge(
                correction_node_id,
                source_node,
                edge_type="user_correction",
                weight=1.0,
            )

        # Prepare a JSON-safe copy for the frontend.
        updated_graph_data = serialize_graph(G, page_sizes)

    # 3. Persist the graph outside the state lock.
    try:
        await asyncio.to_thread(
            persist_graph,
            req.doc_id,
            filename,
            G,
            page_sizes,
            pages,
            elements,
        )
    except Exception as e:
        # Roll back the in-memory addition if persistence fails.
        with state_lock:
            if correction_node_id in G:
                G.remove_node(correction_node_id)
            state["graph_data_json"] = serialize_graph(G, page_sizes)

        print(f"Failed to persist correction: {e}")
        raise HTTPException(
            status_code=500,
            detail=f"Graph update could not be saved: {str(e)}",
        )

    # 4. Publish the saved graph as the current frontend graph.
    with state_lock:
        state["graph_data_json"] = updated_graph_data

    # 5. Record the feedback for auditing. Graph persistence is
    # already complete, so a feedback-log failure must not undo it.
    record = {
        "feedback_id": feedback_id,
        "doc_id": req.doc_id,
        "question": req.question,
        "answer": req.answer,
        "correction": correction,
        "source_ids": [str(n) for n in source_nodes],
        "correction_node_id": correction_node_id,
        "status": "applied",
        "created_at": time.time(),
    }

    try:
        col_feedback.upsert(
            ids=[feedback_id],
            embeddings=[[0.0]],
            documents=[json.dumps(record, ensure_ascii=False)],
            metadatas=[{
                "doc_id": req.doc_id,
                "status": "applied",
            }],
        )
    except Exception as e:
        print(f"Warning: graph saved, but feedback log failed: {e}")

    print(
        f"Correction applied: {correction_node_id}; "
        f"added {len(source_nodes)} correction edge(s); "
        f"graph now has {G.number_of_nodes()} nodes and "
        f"{G.number_of_edges()} edges."
    )

    # 6. Return the actual updated graph, not the old graph.
    return {
        "status": "applied",
        "feedback_id": feedback_id,
        "correction_node_id": correction_node_id,
        "added_links": len(source_nodes),
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "message": (
            f"Correction applied and saved. "
            f"Added {len(source_nodes)} graph link(s)."
        ),
        "graphData": updated_graph_data,
    }


# ============================================================
# LAUNCH SERVER IN GOOGLE COLAB
# ============================================================

nest_asyncio.apply()

# Load the token from Colab Secrets (key: NGROK_AUTH_TOKEN) or an env var.

ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Close any existing tunnel on a repeated cell execution.
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print(f"Backend live at: {public_url}")
print(f"API status: {public_url}/status")

async def run_server():
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
    )
    server = uvicorn.Server(config)
    await server.serve()

await run_server()

# restart runtime

In [ ]:
import os
os._exit(0)